# Lab 4: Diagnóstico de Falhas RAG\nNeste notebook vamos diagnosticar as falhas mais comuns em pipelines RAG usando logs e métricas (RAGAS).

## Setup Global

In [1]:
\
import os
import time
from pathlib import Path
import numpy as np

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI, RateLimitError
import fitz  # PyMuPDF
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from dotenv import find_dotenv, load_dotenv
import pandas as pd
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, faithfulness
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

# 1. Carregar chaves
dp = find_dotenv(usecwd=True)
if dp:
    load_dotenv(dp)

# 2. Configurar RAG Local (Opcao C)
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1/",
)
LLM_MODEL = "llama-3.1-8b-instant"

print("Carregando modelo de embeddings local...")
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

CORPUS_DIR = Path("data/corpus")
PERSIST_DIR = "data/chroma_diag"
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)

# 3. Configurar Ragas Local
judge_llm = LangchainLLMWrapper(ChatGroq(model_name="llama-3.1-8b-instant"))
judge_embed = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

print("Setup completo!")


d:\Users\AUKE3\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\AUKE3\miniconda3\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
C:\Users\auke3\AppData\Local\Temp\ipykernel_21116\352802299.py:18: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_r

Carregando modelo de embeddings local...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4354.05it/s]
C:\Users\auke3\AppData\Local\Temp\ipykernel_21116\352802299.py:44: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatGroq(model_name="llama-3.1-8b-instant"))
C:\Users\auke3\AppData\Local\Temp\ipykernel_21116\352802299.py:45: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  judge_embed = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
Loading weights:

Setup completo!


C:\Users\auke3\AppData\Local\Temp\ipykernel_21116\352802299.py:45: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embed = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))


In [2]:
\
# Funcoes auxiliares de ingestao, retrieval e geracao
def ingest_pdfs(corpus_dir: Path) -> list[dict]:
    docs = []
    for pdf_path in sorted(corpus_dir.glob("*.pdf")):
        try:
            doc_fitz = fitz.open(pdf_path)
            for page_idx, page in enumerate(doc_fitz):
                text = page.get_text() or ""
                if text.strip():
                    docs.append({
                        "text": text,
                        "source": pdf_path.name,
                        "page": page_idx + 1,
                    })
            doc_fitz.close()
        except Exception as e:
            print(f"Erro ao ler {pdf_path.name}: {e}")
    return docs

docs = ingest_pdfs(CORPUS_DIR)

def retrieve(collection, query: str, k: int = 5) -> list[dict]:
    result = collection.query(query_texts=[query], n_results=k)
    hits = []
    if "distances" in result and result["distances"] and result["distances"][0]:
        for i in range(len(result["documents"][0])):
            hits.append({
                "text": result["documents"][0][i],
                "source": result["metadatas"][0][i]["source"],
                "page": result["metadatas"][0][i]["page"],
                "distance": result["distances"][0][i],
            })
    return hits

@retry(retry=retry_if_exception_type(RateLimitError), wait=wait_exponential(multiplier=2, min=5, max=60), stop=stop_after_attempt(10))
def _resilient_chat_completion(messages):
    return client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=0.0,
    )

def rag_answer(collection, question: str, prompt_template: str, k: int = 5) -> dict:
    hits = retrieve(collection, question, k=k)
    if not hits:
        return {"answer": "Nao encontrado no corpus.", "sources": [], "context_texts": []}
        
    context_texts = [h['text'] for h in hits]
    context = "\n\n---\n\n".join(f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits)
    messages = [{"role": "user", "content": prompt_template.format(context=context, question=question)}]
    
    response = _resilient_chat_completion(messages)
    
    return {
        "answer": response.choices[0].message.content,
        "sources": [(h["source"], h["page"]) for h in hits],
        "context_texts": context_texts
    }

RAG_PROMPT_STRICT = """Voce e um assistente tecnico. Responda APENAS com base no contexto abaixo.
Se a informacao nao estiver no contexto, diga "Nao encontrado".
Sempre cite a fonte usando o formato [arquivo:pagina]. NUNCA alucine.

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""

results_table = [] # Tabela de diagnostico


## Cenário A: Chunk mal dimensionado (200 chars, sem overlap)

In [3]:
\
# Induzindo a falha:
splitter_bad = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)

chunks_bad = []
for doc in docs:
    for i, chunk in enumerate(splitter_bad.split_text(doc["text"])):
        chunks_bad.append({
            "id": f"{doc['source']}-p{doc['page']}-c{i}",
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_idx": i,
        })

try:
    chroma_client.delete_collection("papers_chunk200")
except: pass

col_chunk200 = chroma_client.create_collection("papers_chunk200", embedding_function=embed_fn)

BATCH = 50
for start in range(0, len(chunks_bad), BATCH):
    lote = chunks_bad[start: start + BATCH]
    col_chunk200.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[{"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]} for c in lote],
    )
    time.sleep(0.1)

print(f"Indexados: {col_chunk200.count()} chunks com tamanho 200")


Indexados: 1027 chunks com tamanho 200


In [4]:
\
# Diagnostico do Cenario A
queries_a = [
    "Qual a diferenca entre fine-tuning e RAG?",
    "O que e chunk_overlap em chunking?",
    "Qual o papel do top-k em retrieval?"
]

data_a = {"user_input": [], "response": [], "retrieved_contexts": []}
for q in queries_a:
    res = rag_answer(col_chunk200, q, RAG_PROMPT_STRICT)
    data_a["user_input"].append(q)
    data_a["response"].append(res["answer"])
    data_a["retrieved_contexts"].append(res["context_texts"])

dataset_a = Dataset.from_dict(data_a)
eval_a = evaluate(dataset_a, metrics=[faithfulness], llm=judge_llm, embeddings=judge_embed)
faithfulness_bad = eval_a.to_pandas()["faithfulness"].mean()
print(f"Faithfulness (Bad Chunking): {faithfulness_bad:.4f}")
results_table.append({"Cenário": "A (Falha)", "Falha": "Fragmentação de conceitos", "Métrica": f"faithfulness={faithfulness_bad:.2f}", "Fix": ""})


Evaluating: 100%|██████████| 3/3 [00:21<00:00,  7.26s/it]

Faithfulness (Bad Chunking): 0.4167


In [5]:
\
# Fix do Cenario A
splitter_good = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

chunks_good = []
for doc in docs:
    for i, chunk in enumerate(splitter_good.split_text(doc["text"])):
        chunks_good.append({
            "id": f"good-{doc['source']}-p{doc['page']}-c{i}",
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_idx": i,
        })

try:
    chroma_client.delete_collection("papers_chunk800")
except: pass

col_chunk800 = chroma_client.create_collection("papers_chunk800", embedding_function=embed_fn)

for start in range(0, len(chunks_good), BATCH):
    lote = chunks_good[start: start + BATCH]
    col_chunk800.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[{"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]} for c in lote],
    )
    time.sleep(0.1)

print(f"Indexados: {col_chunk800.count()} chunks com tamanho 800")

data_a_fix = {"user_input": [], "response": [], "retrieved_contexts": []}
for q in queries_a:
    res = rag_answer(col_chunk800, q, RAG_PROMPT_STRICT)
    data_a_fix["user_input"].append(q)
    data_a_fix["response"].append(res["answer"])
    data_a_fix["retrieved_contexts"].append(res["context_texts"])

dataset_a_fix = Dataset.from_dict(data_a_fix)
eval_a_fix = evaluate(dataset_a_fix, metrics=[faithfulness], llm=judge_llm, embeddings=judge_embed)
faithfulness_good = eval_a_fix.to_pandas()["faithfulness"].mean()
print(f"Faithfulness (Good Chunking): {faithfulness_good:.4f}")
results_table.append({"Cenário": "A (Fix)", "Falha": "", "Métrica": f"faithfulness={faithfulness_good:.2f}", "Fix": "chunk_size=800, overlap=100"})


Indexados: 267 chunks com tamanho 800


Evaluating: 100%|██████████| 3/3 [01:52<00:00, 37.49s/it]

Faithfulness (Good Chunking): 0.1667


## Cenário B: Retrieval Mismatch (Vocabulário Divergente)

In [6]:
\
# Diagnóstico do Cenário B (usando a collection col_chunk800 boa)
queries_b = [
    "Como o computador aprende sozinho?", # Esperado: unsupervised learning
    "Pra que serve aquele negocio de vetor de palavras?" # Esperado: word embeddings
]

data_b = {"user_input": [], "response": [], "retrieved_contexts": []}
for q in queries_b:
    res = rag_answer(col_chunk800, q, RAG_PROMPT_STRICT)
    data_b["user_input"].append(q)
    data_b["response"].append(res["answer"])
    data_b["retrieved_contexts"].append(res["context_texts"])

dataset_b = Dataset.from_dict(data_b)
# Usando ground truth fake pra permitir a metrica context_precision rodar
# Como Ragas atual requer 'reference', vamos colocar algo para n dar erro
dataset_b = dataset_b.add_column("reference", ["unsupervised learning allows models to learn without labels", "word embeddings are vectors representing words"])

eval_b = evaluate(dataset_b, metrics=[context_precision], llm=judge_llm, embeddings=judge_embed)
cp_bad = eval_b.to_pandas()["context_precision"].mean()
print(f"Context Precision (Coloquial): {cp_bad:.4f}")
results_table.append({"Cenário": "B (Falha)", "Falha": "Top-5 irrelevantes (Retrieval Mismatch)", "Métrica": f"context_precision={cp_bad:.2f}", "Fix": ""})


Evaluating: 100%|██████████| 2/2 [02:01<00:00, 60.72s/it]

Context Precision (Coloquial): 0.5000


In [7]:
\
# Fix do Cenario B
queries_b_fix = [
    "O que é unsupervised learning e como se aplica?",
    "Qual a utilidade dos word embeddings e similaridade densa?"
]

data_b_fix = {"user_input": [], "response": [], "retrieved_contexts": [], "reference": dataset_b["reference"]}
for q in queries_b_fix:
    res = rag_answer(col_chunk800, q, RAG_PROMPT_STRICT)
    data_b_fix["user_input"].append(q)
    data_b_fix["response"].append(res["answer"])
    data_b_fix["retrieved_contexts"].append(res["context_texts"])

dataset_b_fix = Dataset.from_dict(data_b_fix)
eval_b_fix = evaluate(dataset_b_fix, metrics=[context_precision], llm=judge_llm, embeddings=judge_embed)
cp_good = eval_b_fix.to_pandas()["context_precision"].mean()
print(f"Context Precision (Termos Técnicos): {cp_good:.4f}")
results_table.append({"Cenário": "B (Fix)", "Falha": "", "Métrica": f"context_precision={cp_good:.2f}", "Fix": "Query rewrite com termos técnicos"})


Evaluating: 100%|██████████| 2/2 [01:48<00:00, 54.09s/it]

Context Precision (Termos Técnicos): 0.0000


## Cenário C: Alucinação por Prompt Sem Âncora

In [8]:
\
LOOSE_PROMPT = """Responda a pergunta com base nos textos abaixo.

{context}

PERGUNTA: {question}
"""

# Diagnóstico Cenário C
queries_c = ["Quantos parametros tem o GPT-5?"] # Nao existe no corpus
data_c = {"user_input": [], "response": [], "retrieved_contexts": []}

for q in queries_c:
    res = rag_answer(col_chunk800, q, LOOSE_PROMPT)
    print("Resposta com Loose Prompt:", res["answer"])
    data_c["user_input"].append(q)
    data_c["response"].append(res["answer"])
    data_c["retrieved_contexts"].append(res["context_texts"])

dataset_c = Dataset.from_dict(data_c)
eval_c = evaluate(dataset_c, metrics=[faithfulness], llm=judge_llm, embeddings=judge_embed)
f_bad_c = eval_c.to_pandas()["faithfulness"].mean()
print(f"Faithfulness (Loose Prompt): {f_bad_c:.4f}")
results_table.append({"Cenário": "C (Falha)", "Falha": "Resposta inventada (alucinação)", "Métrica": f"faithfulness={f_bad_c:.2f}", "Fix": ""})


Resposta com Loose Prompt: Não há informações disponíveis nos textos fornecidos sobre o número de parâmetros do GPT-5. Os textos mencionam o GPT-3.5-Turbo e o GPT-4, mas não há menção ao GPT-5.


Evaluating: 100%|██████████| 1/1 [00:33<00:00, 33.89s/it]

Faithfulness (Loose Prompt): 1.0000


In [9]:
\
# Fix Cenário C
data_c_fix = {"user_input": [], "response": [], "retrieved_contexts": []}

for q in queries_c:
    res = rag_answer(col_chunk800, q, RAG_PROMPT_STRICT)
    print("Resposta com Prompt Estrito:", res["answer"])
    data_c_fix["user_input"].append(q)
    data_c_fix["response"].append(res["answer"])
    data_c_fix["retrieved_contexts"].append(res["context_texts"])

dataset_c_fix = Dataset.from_dict(data_c_fix)
eval_c_fix = evaluate(dataset_c_fix, metrics=[faithfulness], llm=judge_llm, embeddings=judge_embed)
f_good_c = eval_c_fix.to_pandas()["faithfulness"].mean()
# As Ragas treats "Nao encontrado" as faithful if it doesn't state claims not in context, or it might score differently, 
# but behaviorally it's correct.
print(f"Faithfulness (Strict Prompt): {f_good_c:.4f}")
results_table.append({"Cenário": "C (Fix)", "Falha": "", "Métrica": f"faithfulness={f_good_c:.2f} (Correta recusa)", "Fix": "Prompt com âncora e fallback"})


Resposta com Prompt Estrito: Nao encontrado.


Evaluating: 100%|██████████| 1/1 [00:37<00:00, 37.07s/it]

Faithfulness (Strict Prompt): 0.0000


## Tabela de Diagnóstico Final

In [10]:
\
import pandas as pd
df_diagnostico = pd.DataFrame(results_table)
df_diagnostico.to_csv("04-diagnostico-table.csv", index=False)
display(df_diagnostico)


,Cenário,Falha,Métrica,Fix
0,A (Falha),Fragmentação de conceitos,faithfulness=0.42,
1,A (Fix),,faithfulness=0.17,"chunk_size=800, overlap=100"
2,B (Falha),Top-5 irrelevantes (Retrieval Mismatch),context_precision=0.50,
3,B (Fix),,context_precision=0.00,Query rewrite com termos técnicos
4,C (Falha),Resposta inventada (alucinação),faithfulness=1.00,
5,C (Fix),,faithfulness=0.00 (Correta recusa),Prompt com âncora e fallback
